# Qwen2.5-7B Laravel Coder — LoRA train (GPU T4)

4-bit QLoRA on 800 samples → merge → `/kaggle/working/qwen-laravel-coder/`

In [ ]:
!pip install -q transformers peft trl accelerate datasets sentencepiece bitsandbytes

In [ ]:
import json
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import gc
import tarfile
import time
from pathlib import Path

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
KAGGLE_INPUT = Path("/kaggle/input")
DATASET_SLUG = "laravel-coder-lora-train"

def resolve_input_root() -> Path:
    for p in [
        KAGGLE_INPUT / DATASET_SLUG,
        KAGGLE_INPUT / "datasets" / "bhavingajjar22" / DATASET_SLUG,
        *KAGGLE_INPUT.rglob(DATASET_SLUG),
    ]:
        if p.is_dir():
            return p
    raise FileNotFoundError(f"Dataset not found under {KAGGLE_INPUT}")

for _ in range(36):
    try:
        INPUT_ROOT = resolve_input_root()
        break
    except FileNotFoundError:
        time.sleep(10)
else:
    raise FileNotFoundError("Dataset not mounted")

for tar_path in INPUT_ROOT.glob("*.tar"):
    with tarfile.open(tar_path) as tf:
        tf.extractall(INPUT_ROOT)

TRAIN_FILE = next(
    p for p in [
        INPUT_ROOT / "laravel_train.jsonl",
        INPUT_ROOT / "lora" / "laravel_train.jsonl",
        *INPUT_ROOT.rglob("laravel_train.jsonl"),
    ] if p.is_file()
)

ADAPTER_OUT = Path("/kaggle/working/adapter")
MERGED_OUT = Path("/kaggle/working/qwen-laravel-coder")
OFFLOAD = Path("/kaggle/working/offload")
EPOCHS = 1
MAX_SEQ = 512

assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0))
print(f"Train file: {TRAIN_FILE}")

In [ ]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

rows = load_jsonl(TRAIN_FILE)
dataset = Dataset.from_list([
    {"text": tokenizer.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=False)}
    for r in rows
])
print(f"Samples: {len(dataset)}")

In [ ]:
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, trust_remote_code=True, quantization_config=bnb, device_map={"": 0}
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model = get_peft_model(model, LoraConfig(
    r=4, lora_alpha=8, lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"], bias="none", task_type="CAUSAL_LM",
))
model.print_trainable_parameters()

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=SFTConfig(
        output_dir="/kaggle/working/checkpoints",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        bf16=False,
        fp16=False,
        max_length=MAX_SEQ,
    ),
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()
model.save_pretrained(ADAPTER_OUT)
tokenizer.save_pretrained(ADAPTER_OUT)
print(f"Adapter: {ADAPTER_OUT}")

In [ ]:
del model, trainer
gc.collect()
torch.cuda.empty_cache()

OFFLOAD.mkdir(parents=True, exist_ok=True)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    max_memory={0: "10GiB", "cpu": "28GiB"},
    offload_folder=str(OFFLOAD),
)
peft_model = PeftModel.from_pretrained(base, str(ADAPTER_OUT))
merged = peft_model.merge_and_unload()
MERGED_OUT.mkdir(parents=True, exist_ok=True)
merged.save_pretrained(MERGED_OUT)
tokenizer.save_pretrained(MERGED_OUT)

meta = {
    "base_model": BASE_MODEL,
    "train_samples": len(rows),
    "epochs": EPOCHS,
    "max_seq": MAX_SEQ,
    "lora": {"r": 4, "alpha": 8, "targets": ["q_proj", "v_proj"]},
}
(MERGED_OUT / "training_meta.json").write_text(json.dumps(meta, indent=2))
print(f"Merged model saved: {MERGED_OUT}")